In [2]:
# 1. Импорты и загрузка переменных окружения
import os
from dotenv import load_dotenv
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Загружаем переменные из .env (если есть)
load_dotenv()

print("✅ Библиотеки импортированы")
print(f"Текущая директория: {os.getcwd()}")

✅ Библиотеки импортированы
Текущая директория: d:\Projects\rag-assistant\notebooks


In [3]:
# 2. Переходим в корень проекта
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print(f"✅ Перешли в корень: {os.getcwd()}")
else:
    print(f"✅ Уже в корне: {os.getcwd()}")

# Проверяем наличие папки data
print(f"Папка data существует: {os.path.exists('data')}")

✅ Перешли в корень: d:\Projects\rag-assistant
Папка data существует: True


In [4]:
# 3. Загружаем PDF из папки data
import os
from langchain_community.document_loaders import PyPDFLoader

# Ищем PDF файлы в папке data
pdf_files = [f for f in os.listdir("data") if f.endswith(".pdf")]

if pdf_files:
    file_path = os.path.join("data", pdf_files[0])
    print(f"✅ Найден файл: {pdf_files[0]}")
    
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    print(f"✅ Загружено {len(documents)} страниц")
    print(f"📄 Пример текста: {documents[0].page_content[:200]}...")
else:
    print("❌ В папке data нет PDF файлов")
    print("Положи любой PDF в папку data/ и перезапусти ячейку")

✅ Найден файл: 2301.00234v6.pdf
✅ Загружено 22 страниц
📄 Пример текста: A Survey on In-context Learning
Qingxiu Dong1, Lei Li1, Damai Dai1, Ce Zheng1, Jingyuan Ma1, Rui Li1, Heming Xia2,
Jingjing Xu3, Zhiyong Wu4, Tianyu Liu5, Baobao Chang1, Xu Sun1, Lei Li6 and Zhifang S...


In [5]:
# 4. Разбиваем текст на чанки
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"✅ Создано {len(chunks)} чанков")
print(f"📊 Средний размер: {sum(len(c.page_content) for c in chunks)//len(chunks)} символов")
print(f"📄 Пример чанка: {chunks[0].page_content[:150]}...")

✅ Создано 243 чанков
📊 Средний размер: 456 символов
📄 Пример чанка: A Survey on In-context Learning
Qingxiu Dong1, Lei Li1, Damai Dai1, Ce Zheng1, Jingyuan Ma1, Rui Li1, Heming Xia2,
Jingjing Xu3, Zhiyong Wu4, Tianyu L...


In [6]:
# 5. Создаём эмбеддинги и векторное хранилище
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': False}
)

# Создаём Chroma базу
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db",
    collection_metadata={"hnsw:space": "cosine"}
)

# Сохраняем на диск
vectordb.persist()
print(f"✅ Векторная база создана и сохранена")
print(f"  - Всего векторов: {vectordb._collection.count()}")
print(f"  - Папка: ./chroma_db")

C:\Users\Китя\AppData\Local\Temp\ipykernel_13992\197119219.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 83.57it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Векторная база создана и сохранена
  - Всего векторов: 486
  - Папка: ./chroma_db


C:\Users\Китя\AppData\Local\Temp\ipykernel_13992\197119219.py:20: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [7]:
# 6. Тестируем семантический поиск
query = "Что такое in-context learning?"  # вопрос по содержанию PDF

# Поиск с возвратом релевантности
results = vectordb.similarity_search_with_relevance_scores(query, k=3)

print(f"🔍 Вопрос: {query}\n")
for i, (doc, score) in enumerate(results):
    print(f"\n--- Результат {i+1} (релевантность: {score:.4f}) ---")
    print(doc.page_content[:300] + "...")
    
print(f"\n📊 Всего найдено чанков: {len(results)}")

🔍 Вопрос: Что такое in-context learning?


--- Результат 1 (релевантность: 0.4633) ---
Danny Hernandez, Scott Johnston, Andy Jones, Jack-
son Kernion, Liane Lovitt, Kamal Ndousse, Dario
Amodei, Tom Brown, Jack Clark, Jared Kaplan, Sam
McCandlish, and Chris Olah. 2022. In-context learn-
ing and induction heads. CoRR, abs/2209.11895.
OpenAI. 2023. GPT-4 technical report. CoRR,
abs/2303....

--- Результат 2 (релевантность: 0.4633) ---
Danny Hernandez, Scott Johnston, Andy Jones, Jack-
son Kernion, Liane Lovitt, Kamal Ndousse, Dario
Amodei, Tom Brown, Jack Clark, Jared Kaplan, Sam
McCandlish, and Chris Olah. 2022. In-context learn-
ing and induction heads. CoRR, abs/2209.11895.
OpenAI. 2023. GPT-4 technical report. CoRR,
abs/2303....

--- Результат 3 (релевантность: 0.4411) ---
ested in this area and shed light on future research.
2 Definition and Formulation
Following Brown et al. (2020), we here provide a
formal definition of in-context learning:
In-context learning is a paradigm that
al

In [8]:
# 7. Сохраняем информацию о базе для следующего этапа
import pickle

with open("vectordb_config.pkl", "wb") as f:
    pickle.dump({
        "db_path": "./chroma_db",
        "embedding_model": "all-MiniLM-L6-v2"
    }, f)

print("✅ Конфигурация сохранена в vectordb_config.pkl")
print("🎉 Этап 1 полностью завершён!")
print("\n📊 Итог:")
print(f"  - Загружено страниц: {len(documents)}")
print(f"  - Создано чанков: {len(chunks)}")
print(f"  - Векторов в базе: {vectordb._collection.count()}")
print(f"  - Папка с базой: ./chroma_db")

✅ Конфигурация сохранена в vectordb_config.pkl
🎉 Этап 1 полностью завершён!

📊 Итог:
  - Загружено страниц: 22
  - Создано чанков: 243
  - Векторов в базе: 486
  - Папка с базой: ./chroma_db
